# GroupDNA — WhatsApp Group Chat Analyzer

**Goal:** Read `hostel_bois.txt`, clean and parse the WhatsApp export, calculate group statistics, build a NumPy activity heatmap, analyze response/silent patterns, assign personality archetypes, and print one polished final report.

### Important project restrictions
This notebook follows the project brief:
- Python fundamentals only
- NumPy is allowed
- `datetime` is allowed for timestamps
- No pandas
- No regex
- No `collections.Counter` / `defaultdict`
- No matplotlib / seaborn / plotly
- No ML/NLP libraries

**Input:** `hostel_bois.txt`

## Step 0 — Put the dataset in the same folder

For normal Jupyter Notebook:

```text
your_project_folder/
│
├── GroupDNA.ipynb
└── hostel_bois.txt
```

If the file is in another folder, change `FILE_PATH` in the next cell.

In [ ]:
import numpy as np
from datetime import datetime, timedelta
import string

FILE_PATH = "hostel_bois.txt"

# Read the file
with open(FILE_PATH, "r", encoding="utf-8") as f:
    content = f.read()

lines = content.splitlines()

print("File loaded successfully!")
print("Total raw lines:", len(lines))

## Step 1 — Create the WhatsApp parser

The parser handles system messages, media messages, deleted messages, empty lines, and multi-line messages.

In [ ]:
def looks_like_date(line):
    # Checks DD/MM/YY using string operations only (no regex)
    return (
        len(line) >= 8
        and line[2] == "/"
        and line[5] == "/"
        and line[0:2].isdigit()
        and line[3:5].isdigit()
        and line[6:8].isdigit()
    )


def parse_chat(lines):
    messages = []
    system_count = 0
    media_count = 0
    deleted_count = 0
    current_message = None

    for raw_line in lines:
        line = raw_line.strip()

        if line == "":
            continue

        # New timestamped line
        if looks_like_date(line):
            if " - " not in line:
                system_count += 1
                current_message = None
                continue

            timestamp, rest = line.split(" - ", 1)

            # A normal user message must contain "Sender: message"
            if ": " not in rest:
                system_count += 1
                current_message = None
                continue

            sender, text = rest.split(": ", 1)

            if sender.strip() == "":
                system_count += 1
                current_message = None
                continue

            message = {
                "timestamp": timestamp,
                "sender": sender.strip(),
                "text": text.strip()
            }

            # Keep every sender message for activity statistics.
            # Mark special message types separately.
            if text.strip() == "<Media omitted>":
                message["kind"] = "media"
                media_count += 1
            elif text.strip() == "This message was deleted":
                message["kind"] = "deleted"
                deleted_count += 1
            else:
                message["kind"] = "real"

            messages.append(message)
            current_message = message

        else:
            # Bonus: continuation of a multi-line message
            if current_message is not None:
                current_message["text"] += " " + line

    return messages, system_count, media_count, deleted_count


messages, system_count, media_count, deleted_count = parse_chat(lines)

participants = sorted(set(m["sender"] for m in messages))

print("Successfully parsed", len(messages), "sender messages")
print("Participants:", len(participants))
print("System messages skipped:", system_count)
print("Media messages:", media_count)
print("Deleted messages:", deleted_count)
print("Participants:", participants)

## Step 2 — Convert timestamps and add useful fields

In [ ]:
for message in messages:
    message["dt"] = datetime.strptime(message["timestamp"], "%d/%m/%y, %H:%M")
    message["date"] = message["dt"].date()
    message["hour"] = message["dt"].hour

# Sort by timestamp to make all later calculations reliable
messages.sort(key=lambda x: x["dt"])

print("First 5 parsed messages:")
for m in messages[:5]:
    print(m)

print("\nLast 5 parsed messages:")
for m in messages[-5:]:
    print(m)

## Step 3 — Group Overview

This section calculates total messages, date range, participants, and messages per person.

In [ ]:
start_date = min(m["date"] for m in messages)
end_date = max(m["date"] for m in messages)
total_days = (end_date - start_date).days + 1

message_counts = {}

for person in participants:
    message_counts[person] = sum(1 for m in messages if m["sender"] == person)

print("=" * 60)
print("GROUP OVERVIEW")
print("=" * 60)
print("Group          : Hostel Bois 4ever")
print(f"Period         : {start_date.strftime('%d %B %Y')} to {end_date.strftime('%d %B %Y')} ({total_days} days)")
print(f"Total messages : {len(messages):,}")
print(f"Participants   : {len(participants)}")

print("\nMESSAGES PER PERSON")
for person, count in sorted(message_counts.items(), key=lambda x: x[1], reverse=True):
    percentage = count / len(messages) * 100
    print(f"{person:<10} : {count:>4} ({percentage:5.1f}%)")

## Step 4 — Busiest Day and Busiest Hour

In [ ]:
day_counts = {}
hour_counts = {}

for m in messages:
    day_counts[m["date"]] = day_counts.get(m["date"], 0) + 1
    hour_counts[m["hour"]] = hour_counts.get(m["hour"], 0) + 1

busiest_day = max(day_counts, key=day_counts.get)
busiest_hour = max(hour_counts, key=hour_counts.get)

print("=" * 60)
print("MOST ACTIVE DAY AND HOUR")
print("=" * 60)
print(
    f"Busiest day  : {busiest_day.strftime('%d %B %Y')} "
    f"({day_counts[busiest_day]} messages)"
)
print(
    f"Busiest hour : {busiest_hour:02d}:00 - {(busiest_hour + 1) % 24:02d}:00 "
    f"({hour_counts[busiest_hour]} messages)"
)
print(f"Average/day  : {len(messages) / total_days:.1f} messages")

## Step 5 — NumPy Activity Heatmap

Rows = participants. Columns = hours 0–23. Each cell stores the number of messages sent by that person in that hour.

In [ ]:
person_index = {person: i for i, person in enumerate(participants)}

heatmap = np.zeros((len(participants), 24), dtype=int)

for m in messages:
    row = person_index[m["sender"]]
    col = m["hour"]
    heatmap[row, col] += 1

print("Heatmap shape:", heatmap.shape)
print("Rows:", participants)
print("Columns: hours 0 to 23")

In [ ]:
def shade_for_value(value, maximum):
    if value == 0 or maximum == 0:
        return "."
    ratio = value / maximum

    if ratio <= 0.25:
        return "░"
    elif ratio <= 0.50:
        return "▒"
    else:
        return "█"


print("=" * 90)
print("ACTIVITY HEATMAP — messages by hour")
print("=" * 90)

print("Person     " + " ".join(f"{h:02d}" for h in range(24)))

for i, person in enumerate(participants):
    row = heatmap[i]
    maximum = np.max(row)

    blocks = [shade_for_value(value, maximum) for value in row]
    print(f"{person:<10} " + "  ".join(blocks))

## Step 6 — Top Words

Only normal text messages are used. Media and deleted messages are excluded from word analysis.

In [ ]:
stop_words = {
    "i", "is", "the", "a", "an", "and", "or", "to", "of",
    "in", "on", "for", "it", "this", "that", "you", "me",
    "my", "we", "are", "was", "be", "with", "so", "just",
    "have", "has", "had", "but", "at", "from", "they", "he",
    "she", "do", "did", "not", "its"
}

def clean_words(text):
    cleaned = text.lower()

    # Replace punctuation with spaces
    for symbol in string.punctuation:
        cleaned = cleaned.replace(symbol, " ")

    return cleaned.split()


word_counts = {}

for m in messages:
    if m["kind"] != "real":
        continue

    for word in clean_words(m["text"]):
        if word == "" or word in stop_words:
            continue
        word_counts[word] = word_counts.get(word, 0) + 1

top_words = sorted(word_counts.items(), key=lambda x: x[1], reverse=True)[:10]

print("=" * 60)
print("THIS GROUP'S FAVOURITE WORDS")
print("=" * 60)

max_word_count = top_words[0][1] if top_words else 1

for word, count in top_words:
    bar_length = max(1, int(count / max_word_count * 25))
    print(f"{word:<12} {'█' * bar_length:<25} {count}")

### Bonus — Top 5 words for every person

In [ ]:
print("=" * 60)
print("TOP 5 WORDS PER PERSON")
print("=" * 60)

for person in participants:
    personal_words = {}

    for m in messages:
        if m["sender"] != person or m["kind"] != "real":
            continue

        for word in clean_words(m["text"]):
            if word == "" or word in stop_words:
                continue
            personal_words[word] = personal_words.get(word, 0) + 1

    top5 = sorted(personal_words.items(), key=lambda x: x[1], reverse=True)[:5]
    print(f"\n{person}:")
    for word, count in top5:
        print(f"  {word:<12} {count}")

## Step 7 — Message Length Statistics

In [ ]:
average_words = {}

for person in participants:
    normal_messages = [
        m for m in messages
        if m["sender"] == person and m["kind"] == "real"
    ]

    if normal_messages:
        total_words = sum(len(m["text"].split()) for m in normal_messages)
        average_words[person] = total_words / len(normal_messages)
    else:
        average_words[person] = 0

print("=" * 60)
print("AVERAGE MESSAGE LENGTH")
print("=" * 60)

for person in sorted(participants, key=lambda p: average_words[p], reverse=True):
    print(f"{person:<10} : {average_words[person]:.1f} words/message")

## Step 8 — Response Speed

Definition used: when a person sends a message immediately after another participant's message, the time gap is treated as that person's response gap.

In [ ]:
response_gaps = {person: [] for person in participants}

for i in range(1, len(messages)):
    previous = messages[i - 1]
    current = messages[i]

    if previous["sender"] != current["sender"]:
        gap_seconds = (current["dt"] - previous["dt"]).total_seconds()

        # Ignore impossible/negative gaps just in case the input is malformed.
        if gap_seconds >= 0:
            response_gaps[current["sender"]].append(gap_seconds)

average_response_minutes = {}

for person in participants:
    gaps = response_gaps[person]

    if gaps:
        average_response_minutes[person] = np.mean(gaps) / 60
    else:
        average_response_minutes[person] = None

print("=" * 60)
print("RESPONSE PATTERNS")
print("=" * 60)

for person in sorted(
    participants,
    key=lambda p: average_response_minutes[p] if average_response_minutes[p] is not None else float("inf")
):
    value = average_response_minutes[person]

    if value is None:
        print(f"{person:<10} : No response gaps available")
    elif value < 60:
        print(f"{person:<10} : {value:.1f} minutes")
    else:
        print(f"{person:<10} : {value / 60:.1f} hours")

## Step 9 — Longest Silent Streak

A silent day is a calendar day in the chat period on which a participant sent zero messages. The longest consecutive run of such days is reported.

In [ ]:
all_dates = []
current_date = start_date

while current_date <= end_date:
    all_dates.append(current_date)
    current_date += timedelta(days=1)


def longest_silent_streak(person):
    active_dates = {m["date"] for m in messages if m["sender"] == person}

    longest = 0
    current = 0
    streak_start = None
    best_start = None
    best_end = None

    for date in all_dates:
        if date not in active_dates:
            if current == 0:
                streak_start = date

            current += 1

            if current > longest:
                longest = current
                best_start = streak_start
                best_end = date
        else:
            current = 0

    return longest, best_start, best_end


silent_streaks = {}

for person in participants:
    silent_streaks[person] = longest_silent_streak(person)

print("=" * 60)
print("LONGEST SILENT STREAKS")
print("=" * 60)

for person in sorted(
    participants,
    key=lambda p: silent_streaks[p][0],
    reverse=True
):
    length, s, e = silent_streaks[person]

    if length == 0:
        print(f"{person:<10} : 0 days")
    elif length == 1:
        print(f"{person:<10} : 1 day ({s.strftime('%d %b')} only)")
    else:
        print(
            f"{person:<10} : {length} days "
            f"({s.strftime('%d %b')} to {e.strftime('%d %b')})"
        )

## Step 10 — Calculate all archetype metrics

The eight required archetypes are: Spammer, Group Mom, Night Owl, Storyteller, Drama Queen, Ghost, Comedian, and Question Master.

In [ ]:
caring_keywords = [
    "okay", "safe", "eat", "sleep", "take care",
    "are you", "please", "reminder", "drink water", "don't forget"
]

comedy_keywords = ["lol", "lmao", "haha", "rofl", "lmfao"]


def average_burst_for_person(person):
    bursts = []
    last_sender = None
    run_length = 0

    for m in messages:
        if m["sender"] == last_sender:
            run_length += 1
        else:
            if last_sender == person:
                bursts.append(run_length)
            last_sender = m["sender"]
            run_length = 1

    if last_sender == person:
        bursts.append(run_length)

    return np.mean(bursts) if bursts else 0


def caring_keyword_count(person):
    total = 0

    for m in messages:
        if m["sender"] != person:
            continue

        lower_text = m["text"].lower()

        for keyword in caring_keywords:
            total += lower_text.count(keyword)

    return total


def night_owl_ratio(person):
    person_messages = [m for m in messages if m["sender"] == person]

    if not person_messages:
        return 0

    night_messages = sum(
        1 for m in person_messages
        if m["hour"] >= 23 or m["hour"] <= 4
    )

    return night_messages / len(person_messages)


def storyteller_score(person):
    normal = [
        m for m in messages
        if m["sender"] == person and m["kind"] == "real"
    ]

    if not normal:
        return 0

    return np.mean([len(m["text"].split()) for m in normal])


def drama_score(person):
    person_messages = [m for m in messages if m["sender"] == person]

    if not person_messages:
        return 0

    uppercase_ratio = sum(
        1 for m in person_messages
        if len(m["text"]) >= 3 and m["text"].isupper()
    ) / len(person_messages)

    exclamation_ratio = sum(
        1 for m in person_messages
        if m["text"].count("!") >= 2
    ) / len(person_messages)

    return max(uppercase_ratio, exclamation_ratio)


def ghost_score(person):
    active_days = len({
        m["date"] for m in messages
        if m["sender"] == person
    })

    silent_days = total_days - active_days
    return silent_days / total_days


def comedian_score(person):
    person_messages = [m for m in messages if m["sender"] == person]

    if not person_messages:
        return 0

    comedy_count = 0

    for m in person_messages:
        lower_text = m["text"].lower()
        for keyword in comedy_keywords:
            comedy_count += lower_text.count(keyword)

    return comedy_count / len(person_messages)


def question_master_score(person):
    person_messages = [m for m in messages if m["sender"] == person]

    if not person_messages:
        return 0

    questions = sum(
        1 for m in person_messages
        if m["text"].endswith("?")
    )

    return questions / len(person_messages)


metrics = {
    person: {
        "spammer": average_burst_for_person(person),
        "group_mom": caring_keyword_count(person),
        "night_owl": night_owl_ratio(person),
        "storyteller": storyteller_score(person),
        "drama_queen": drama_score(person),
        "ghost": ghost_score(person),
        "comedian": comedian_score(person),
        "question_master": question_master_score(person)
    }
    for person in participants
}

print("Archetype metrics calculated.")

## Step 11 — Assign exactly one archetype per person

Different archetypes use different units, so we normalize each metric against the group maximum. A score is zero when the person does not cross that archetype's required threshold. Ties are resolved using the order written below; this is documented so the assignment is deterministic.

In [ ]:
archetype_names = {
    "spammer": "THE SPAMMER",
    "group_mom": "THE GROUP MOM",
    "night_owl": "THE NIGHT OWL",
    "storyteller": "THE STORYTELLER",
    "drama_queen": "THE DRAMA QUEEN",
    "ghost": "THE GHOST",
    "comedian": "THE COMEDIAN",
    "question_master": "THE QUESTION MASTER"
}

# Thresholds from the project brief.
def eligible(person, archetype):
    value = metrics[person][archetype]

    if archetype == "spammer":
        return value > 3
    if archetype == "group_mom":
        return value > 0
    if archetype == "night_owl":
        return value > 0.60
    if archetype == "storyteller":
        return value > 30
    if archetype == "drama_queen":
        return value > 0.30
    if archetype == "ghost":
        return value > 0.60
    if archetype == "comedian":
        return value > 0
    if archetype == "question_master":
        return value > 0.25

    return False


def normalized_scores(person):
    scores = {}

    for archetype in archetype_names:
        if not eligible(person, archetype):
            scores[archetype] = 0
            continue

        values = [metrics[p][archetype] for p in participants]
        maximum = max(values)

        if maximum == 0:
            scores[archetype] = 0
        else:
            scores[archetype] = metrics[person][archetype] / maximum * 100

    return scores


archetype_results = {}

# Tie-break rule: earlier archetype in this list wins.
tie_break_order = [
    "spammer",
    "group_mom",
    "night_owl",
    "storyteller",
    "drama_queen",
    "ghost",
    "comedian",
    "question_master"
]

for person in participants:
    scores = normalized_scores(person)

    best_archetype = tie_break_order[0]
    best_score = scores[best_archetype]

    for archetype in tie_break_order[1:]:
        if scores[archetype] > best_score:
            best_archetype = archetype
            best_score = scores[archetype]

    archetype_results[person] = {
        "archetype": best_archetype,
        "score": best_score,
        "all_scores": scores
    }

print("=" * 60)
print("PERSONALITY ARCHETYPES")
print("=" * 60)

for person in sorted(participants):
    result = archetype_results[person]
    print(f"{person:<10} → {archetype_names[result['archetype']]} ({result['score']:.1f})")

## Step 12 — Optional bonus archetype: The Late-Night Philosopher

In [ ]:
# Bonus archetype: specifically designed around Indian-college-style late-night
# philosophical messages. It is not used to replace the required eight archetypes;
# it is shown as an additional insight.

philosophy_words = ["life", "meaning", "matter", "time", "thoughts", "dread"]


def late_night_philosopher_score(person):
    person_messages = [
        m for m in messages
        if m["sender"] == person
        and (m["hour"] >= 23 or m["hour"] <= 4)
    ]

    if not person_messages:
        return 0

    matches = 0

    for m in person_messages:
        lower_text = m["text"].lower()
        for word in philosophy_words:
            matches += lower_text.count(word)

    return matches / len(person_messages)


bonus_scores = {
    person: late_night_philosopher_score(person)
    for person in participants
}

print("=" * 60)
print("BONUS — LATE-NIGHT PHILOSOPHER")
print("=" * 60)

for person in sorted(bonus_scores, key=bonus_scores.get, reverse=True):
    print(f"{person:<10} : {bonus_scores[person]:.3f}")

## Step 13 — Final polished report

Run this cell after all previous cells. This is the main screenshot-ready output.

In [ ]:
def format_duration(minutes):
    if minutes is None:
        return "N/A"
    if minutes < 60:
        return f"{minutes:.1f} min"
    return f"{minutes / 60:.1f} hr"


print("\n")
print("┌" + "─" * 68 + "┐")
print("│" + 'GROUPDNA REPORT — "Hostel Bois 4ever"'.center(68) + "│")
print("│" + "WhatsApp Group Chat Analytics".center(68) + "│")
print("└" + "─" * 68 + "┘")

print("\nGROUP SNAPSHOT")
print("─" * 70)
print(f"Period        : {start_date.strftime('%d %B %Y')} → {end_date.strftime('%d %B %Y')}")
print(f"Messages      : {len(messages):,}")
print(f"Participants  : {len(participants)}")
print(f"Active days   : {total_days}")
print(f"Busiest day   : {busiest_day.strftime('%d %B %Y')} ({day_counts[busiest_day]} messages)")
print(f"Busiest hour  : {busiest_hour:02d}:00–{(busiest_hour + 1) % 24:02d}:00 ({hour_counts[busiest_hour]} messages)")

print("\nMESSAGES PER PERSON")
print("─" * 70)
for person, count in sorted(message_counts.items(), key=lambda x: x[1], reverse=True):
    pct = count / len(messages) * 100
    print(f"{person:<10} {count:>4}  ({pct:5.1f}%)")

print("\nTOP WORDS")
print("─" * 70)
for word, count in top_words:
    bar = "█" * max(1, int(count / max_word_count * 25))
    print(f"{word:<12} {bar:<25} {count}")

print("\nRESPONSE PATTERNS")
print("─" * 70)
for person in sorted(
    participants,
    key=lambda p: average_response_minutes[p]
    if average_response_minutes[p] is not None else float("inf")
):
    print(f"{person:<10} {format_duration(average_response_minutes[person])}")

print("\nSILENT STREAKS")
print("─" * 70)
for person in sorted(participants, key=lambda p: silent_streaks[p][0], reverse=True):
    length, s, e = silent_streaks[person]
    if length == 0:
        print(f"{person:<10} 0 days")
    else:
        print(f"{person:<10} {length} days")

print("\nPERSONALITY ARCHETYPES")
print("─" * 70)
for person in sorted(participants):
    result = archetype_results[person]
    archetype = result["archetype"]
    print(f"{person:<10} → {archetype_names[archetype]}")

print("\nDATA QUALITY")
print("─" * 70)
print(f"System messages skipped : {system_count}")
print(f"Media messages          : {media_count}")
print(f"Deleted messages        : {deleted_count}")
print("Multi-line messages     : Supported by parser")
print("Empty lines             : Skipped")

print("\n" + "=" * 70)
print("GROUPDNA COMPLETE")
print("=" * 70)

## Step 14 — Final checks before submission

Run **Kernel → Restart & Run All**.

Your notebook should finish without errors.

### Check these important results
- Parsed sender messages should be **3,174**
- Participants should be **6**
- System messages should be **4**
- Media messages should be around **30+**
- Deleted messages should be around **15**
- Rahul should show the strongest spam-burst pattern
- Aman should have a very high late-night percentage
- Karan should have the highest average message length
- Neha should have a high uppercase/drama percentage
- Vikas should have the largest silent streak
- Priya should have the strongest caring-keyword pattern

### Submission structure

```text
GroupDNA_Project/
│
├── GroupDNA.ipynb
└── hostel_bois.txt
```

### For GitHub
1. Create a new GitHub repository.
2. Upload `GroupDNA.ipynb`.
3. Upload `hostel_bois.txt` if your course requires the dataset.
4. Open the notebook on GitHub and verify it is readable.
5. Keep the notebook's first markdown cell as the project description.

### Important
Do not add pandas, regex, matplotlib, seaborn, plotly, scikit-learn, NLTK, or another WhatsApp analyzer library because the project brief explicitly forbids them.